# 03 - Full Training Pipeline

Goal: run the final project pipeline end to end. This is the notebook that can take hours. Run the smoke notebook first.

In [ ]:
from pathlib import Path
import sys

def _add_project_root_to_path():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "soccer-twos-starter"):
            if (candidate / "soccer_twos_project" / "notebook_tools.py").exists():
                if str(candidate) not in sys.path:
                    sys.path.insert(0, str(candidate))
                return candidate
    raise FileNotFoundError("Could not find the soccer-twos-starter project root.")

_add_project_root_to_path()

import importlib
import soccer_twos_project.notebook_tools as notebook_tools
importlib.reload(notebook_tools)
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

## Configuration

Fill author metadata before exporting. Keep `FULL_TIMESTEPS = None` to use the selected hardware profile default.

In [ ]:
PROFILE_NAME = "auto"  # auto, laptop_cuda, laptop_mps, free_gpu, pro_gpu, cpu_debug
FULL_TIMESTEPS = None
AUTHOR = "Your Name"
EMAIL = "your.email@gatech.edu"
TEAM_AGENT_NAME = "TEAMNAME_AGENT"

from soccer_twos_project.config import profile_dict, select_profile
print_json(profile_dict(select_profile(PROFILE_NAME)))

## TensorBoard

Use this while training to monitor `episode_reward_mean`, loss, entropy, and throughput.

In [ ]:
print("TensorBoard logdir:", ctx.dirs["checkpoints"])
%reload_ext tensorboard
%tensorboard --logdir ./artifacts/cs8803_soccer_twos/checkpoints --reload_interval 10

## Train Agent 1: PPO Baseline

This is the baseline curve for the report.

In [ ]:
ppo_baseline_checkpoint = run_training(ctx, "ppo_baseline", profile_name=PROFILE_NAME, timesteps=FULL_TIMESTEPS, verbose=1)
ppo_baseline_checkpoint

## Train Agent 2: Reward-Shaped PPO

This is the reward-modification experiment for the rubric.

In [ ]:
ppo_shaped_checkpoint = run_training(ctx, "ppo_shaped", profile_name=PROFILE_NAME, timesteps=FULL_TIMESTEPS, verbose=1)
ppo_shaped_checkpoint

## Train Agent 3: Curriculum PPO

This is the main performance candidate.

In [ ]:
ppo_curriculum_checkpoint = run_training(ctx, "ppo_curriculum", profile_name=PROFILE_NAME, timesteps=FULL_TIMESTEPS, verbose=1)
ppo_curriculum_checkpoint

## Optional Fallback: Self-Play PPO

Run only if curriculum fails to beat the baseline. This trains with archived opponent policies and can take longer.

In [ ]:
# ppo_selfplay_checkpoint = run_training(ctx, "ppo_selfplay", profile_name=PROFILE_NAME, timesteps=FULL_TIMESTEPS, verbose=1)
# ppo_selfplay_checkpoint

## Plot Learning Curves

This writes per-run PNGs and one overlaid comparison plot to `artifacts/cs8803_soccer_twos/plots`.

In [ ]:
from soccer_twos_project.plotting import plot_results

plot_results(SimpleNamespace(
    artifact_root=str(ctx.artifact_root),
    ray_results=str(ctx.dirs["checkpoints"]),
    output_dir=str(ctx.dirs["plots"]),
    filter=None,
))

## Export Standalone Agents

Exports are lightweight `AgentInterface` packages. They do not require Ray at evaluation time.

In [ ]:
from soccer_twos_project.exporting import export_checkpoint

def export_agent(stage, agent_name, description):
    export_checkpoint(SimpleNamespace(
        checkpoint=best_checkpoint(ctx, stage),
        stage=stage,
        policy_id="default_policy",
        profile="cpu_debug",
        artifact_root=str(ctx.artifact_root),
        output_dir=None,
        agent_name=agent_name,
        author=AUTHOR,
        email=EMAIL,
        description=description,
        no_zip=False,
        clean=True,
    ))

export_agent("ppo_baseline", "soccer_ppo_baseline", "PPO baseline trained against a still policy opponent.")
export_agent("ppo_shaped", "soccer_ppo_shaped", "PPO with clipped distance-based reward shaping.")
export_agent("ppo_curriculum", "soccer_ppo_curriculum", "PPO trained with curriculum start states.")
# export_agent("ppo_selfplay", "soccer_ppo_selfplay", "PPO self-play fallback with archived opponents.")

## Imitation Agent

Use the strongest exported policy as the expert. If you downloaded `ceia_baseline_agent`, you can set `EXPERT_MODULE = "ceia_baseline_agent"` instead.

In [ ]:
from soccer_twos_project.imitation import collect_dataset, train_bc

EXPERT_MODULE = "soccer_ppo_curriculum"
DATASET_PATH = ctx.dirs["datasets"] / "bc_expert_dataset.npz"

collect_dataset(SimpleNamespace(
    expert_module=EXPERT_MODULE,
    samples=50_000,
    output=str(DATASET_PATH),
    base_port=None,
    artifact_root=str(ctx.artifact_root),
))

train_bc(SimpleNamespace(
    dataset=str(DATASET_PATH),
    agent_name="soccer_bc_imitation",
    author=AUTHOR,
    email=EMAIL,
    description="Behavior cloning agent trained from an expert SoccerTwos policy.",
    hidden_layers="256,256",
    epochs=20,
    batch_size=256,
    lr=1e-3,
    val_fraction=0.1,
    seed=0,
    output_dir=None,
    artifact_root=str(ctx.artifact_root),
    no_zip=False,
))

## Quick Evaluations

Run 5 episodes first to catch import/action bugs, then run 100 episodes in the submission notebook.

In [ ]:
from soccer_twos_project.evaluation import evaluate, safe_label, write_outputs

def evaluate_pair(agent1, agent2, episodes=5):
    rows, summary = evaluate(agent1, agent2, episodes=episodes, base_port=None)
    write_outputs(rows, summary, ctx.dirs["evals"], safe_label(agent1, agent2))
    print_json(summary)
    return summary

evaluate_pair("soccer_ppo_baseline", "soccer_ppo_curriculum", episodes=5)
# evaluate_pair("soccer_ppo_curriculum", "ceia_baseline_agent", episodes=5)